# 01 — Triplet Parsing (CholecT50)

**Goal:** Load the CholecT50 JSON annotations, understand the binary-to-label mapping,
and parse the surgical action triplets `(instrument, verb, target)` for 3 sample videos.

**Videos:** VID01 (short), VID05 (medium), VID40 (longer)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
from src.triplet_parser import (
    load_categories,
    print_categories,
    parse_video,
    parse_multiple_videos,
    filter_valid_triplets,
    multi_label_analysis,
    video_summary,
    save_parsed_csv,
)

# Project paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'data' / 'CholecT45' / 'triplet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Triplet dir:', TRIPLET_DIR)
print('Available:', os.listdir(TRIPLET_DIR))

Triplet dir: /Users/maheshkundurthi/Documents/Research work OPT/causal-knowledge-graph-surgical-ai/data/CholecT45/triplet
Available: ['VID05.json', 'VID40.json', 'VID01.json']


## 1. Inspect Category Dictionaries

Categories are embedded in each video JSON: **6 instruments, 10 verbs, 15 targets, 100 triplet classes**.

In [2]:
categories = load_categories(TRIPLET_DIR / 'VID01.json')
print_categories(categories)


=== INSTRUMENTS (6 classes) ===
  0: grasper
  1: bipolar
  2: hook
  3: scissors
  4: clipper
  5: irrigator

=== VERBS (10 classes) ===
  0: grasp
  1: retract
  2: dissect
  3: coagulate
  4: clip
  5: cut
  6: aspirate
  7: irrigate
  8: pack
  9: null_verb

=== TARGETS (15 classes) ===
  0: gallbladder
  1: cystic_plate
  2: cystic_duct
  3: cystic_artery
  4: cystic_pedicle
  5: blood_vessel
  6: fluid
  7: abdominal_wall_cavity
  8: liver
  9: adhesion
  10: omentum
  11: peritoneum
  12: gut
  13: specimen_bag
  14: null_target

=== TRIPLETS (100 classes) ===
  0: grasper,dissect,cystic_plate
  1: grasper,dissect,gallbladder
  2: grasper,dissect,omentum
  3: grasper,grasp,cystic_artery
  4: grasper,grasp,cystic_duct
  5: grasper,grasp,cystic_pedicle
  6: grasper,grasp,cystic_plate
  7: grasper,grasp,gallbladder
  8: grasper,grasp,gut
  9: grasper,grasp,liver
  ... (100 total)


## 2. Parse a Single Video (VID01)

`parse_video()` decodes each 15-element annotation array into `(instrument, verb, target)`.
Multi-label frames (multiple simultaneous triplets) are fully expanded — one row per triplet.

In [3]:
df_vid01_raw = parse_video(TRIPLET_DIR / 'VID01.json')
print(f'VID01 raw: {len(df_vid01_raw)} rows across {df_vid01_raw["frame"].nunique()} frames')
df_vid01_raw.head(10)

VID01 raw: 2751 rows across 1734 frames


,video,frame,timestamp_sec,entry_index,num_triplets_in_frame,is_multi_label,triplet_id,triplet_label,instrument_id,instrument,verb_id,verb,target_id,target,is_valid,is_null
0,VID01,0,0.0,0,1,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False
1,VID01,1,1.0,0,1,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False
2,VID01,2,2.0,0,1,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False
3,VID01,3,3.0,0,1,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False
4,VID01,4,4.0,0,1,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False
5,VID01,5,5.0,0,1,False,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,False,False
6,VID01,6,6.0,0,1,False,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,False,False
7,VID01,7,7.0,0,1,False,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,False,False
8,VID01,8,8.0,0,1,False,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,False,False
9,VID01,9,9.0,0,1,False,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,-1,unknown_-1,False,False


## 3. Handle Multi-Label Frames

Some frames have **no valid annotation** (triplet_id = -1). We filter these out
and recalculate per-frame counts.

In [4]:
df_vid01 = filter_valid_triplets(df_vid01_raw)
print(f'VID01 valid: {len(df_vid01)} rows across {df_vid01["frame"].nunique()} frames')
print(f'Removed {len(df_vid01_raw) - len(df_vid01)} null entries')
df_vid01.head(10)

VID01 valid: 2560 rows across 1543 frames
Removed 191 null entries


,video,frame,timestamp_sec,entry_index,is_multi_label,triplet_id,triplet_label,instrument_id,instrument,verb_id,verb,target_id,target,is_valid,is_null,num_triplets_in_frame
0,VID01,0,0.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
1,VID01,1,1.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
2,VID01,2,2.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
3,VID01,3,3.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
4,VID01,4,4.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
5,VID01,10,10.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
6,VID01,11,11.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
7,VID01,12,12.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
8,VID01,13,13.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
9,VID01,14,14.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1


In [5]:
# Multi-label analysis
stats = multi_label_analysis(df_vid01)
print('=== Multi-Label Frame Analysis (VID01) ===')
for k, v in stats.items():
    print(f'  {k}: {v}')

=== Multi-Label Frame Analysis (VID01) ===
  total_frames: 1543
  single_label_frames: 715
  multi_label_frames: 828
  multi_label_pct: 53.7
  max_triplets_per_frame: 3
  mean_triplets_per_frame: 1.66
  distribution: {1: 715, 2: 639, 3: 189}


In [6]:
# Show example multi-label frames
multi_frames = df_vid01[df_vid01['is_multi_label']]
if not multi_frames.empty:
    sample_frame = multi_frames['frame'].iloc[0]
    print(f'=== Example Multi-Label Frame {sample_frame} ===')
    display(df_vid01[df_vid01['frame'] == sample_frame][['frame', 'instrument', 'verb', 'target', 'triplet_label']])

=== Example Multi-Label Frame 20 ===


,frame,instrument,verb,target,triplet_label
15,20,grasper,grasp,gallbladder,"grasper,grasp,gallbladder"
16,20,hook,null_verb,null_target,"hook,null_verb,null_target"


## 4. Parse All 3 Sample Videos

Parse VID01, VID05, VID40 together and generate combined statistics.

In [7]:
print('Parsing 3 sample videos...')
df_all_raw = parse_multiple_videos(TRIPLET_DIR, video_ids=['VID01', 'VID05', 'VID40'])
df_all = filter_valid_triplets(df_all_raw)

print(f'\nCombined: {len(df_all)} valid rows, {len(df_all_raw) - len(df_all)} null entries removed')
print(f'Total frames: {df_all.groupby("video")["frame"].nunique().sum()}')

Parsing 3 sample videos...
  Parsed VID01: 2751 rows, 1734 frames
  Parsed VID05: 3459 rows, 2345 frames
  Parsed VID40: 3722 rows, 2223 frames

Combined: 8883 valid rows, 1049 null entries removed
Total frames: 5253


In [8]:
# Per-video summary table
summary = video_summary(df_all)
display(summary)

,video,total_frames,total_triplet_rows,unique_triplets,multi_label_frames,multi_label_pct,mean_triplets_per_frame
0,VID01,1543,2560,22,828,53.7,1.66
1,VID05,1797,2911,26,1075,59.8,1.62
2,VID40,1913,3412,20,1440,75.3,1.78


In [9]:
# Multi-label analysis across all 3 videos
stats_all = multi_label_analysis(df_all)
print('=== Combined Multi-Label Analysis ===')
for k, v in stats_all.items():
    print(f'  {k}: {v}')

=== Combined Multi-Label Analysis ===
  total_frames: 5253
  single_label_frames: 1910
  multi_label_frames: 3343
  multi_label_pct: 63.6
  max_triplets_per_frame: 3
  mean_triplets_per_frame: 1.69
  distribution: {1: 1910, 2: 3056, 3: 287}


## 5. Label Frequency Summary

In [10]:
print('=== Combined Label Frequencies ===')
print('\nTop 15 Triplets:')
print(df_all['triplet_label'].value_counts().head(15))
print('\nInstruments:')
print(df_all['instrument'].value_counts())
print('\nVerbs:')
print(df_all['verb'].value_counts())
print('\nTargets:')
print(df_all['target'].value_counts())

=== Combined Label Frequencies ===

Top 15 Triplets:
triplet_label
grasper,retract,gallbladder      2578
hook,dissect,gallbladder         1305
grasper,grasp,gallbladder         749
grasper,retract,liver             711
hook,dissect,cystic_duct          465
hook,null_verb,null_target        415
grasper,dissect,gallbladder       336
grasper,grasp,specimen_bag        305
grasper,null_verb,null_target     288
irrigator,aspirate,fluid          246
clipper,clip,cystic_duct          221
hook,dissect,cystic_artery        185
hook,dissect,cystic_plate         165
grasper,retract,omentum           164
hook,dissect,omentum              154
Name: count, dtype: int64

Instruments:
instrument
grasper      5247
hook         2731
clipper       310
irrigator     284
bipolar       222
scissors       89
Name: count, dtype: int64

Verbs:
verb
retract      3550
dissect      2618
grasp        1070
null_verb     757
clip          286
aspirate      246
coagulate     228
cut            72
pack           45
irr

## 6. Save Parsed Triplets to CSV

In [11]:
# Save per-video CSVs
print('Saving per-video CSVs...')
saved_files = save_parsed_csv(df_all, OUTPUT_DIR, per_video=True)

# Also save combined
print('\nSaving combined CSV...')
save_parsed_csv(df_all, OUTPUT_DIR, per_video=False)

print(f'\nAll outputs in: {OUTPUT_DIR}')
print('Files:', os.listdir(OUTPUT_DIR))

Saving per-video CSVs...
  Saved VID01_triplets.csv (2560 rows)
  Saved VID05_triplets.csv (2911 rows)
  Saved VID40_triplets.csv (3412 rows)

Saving combined CSV...
  Saved all_triplets.csv (8883 rows)

All outputs in: /Users/maheshkundurthi/Documents/Research work OPT/causal-knowledge-graph-surgical-ai/outputs/parsed_triplets
Files: ['VID05_triplets.csv', 'VID01_triplets.csv', 'VID40_triplets.csv', 'all_triplets.csv']


---
**Next notebook:** `02_graph_construction.ipynb` — build causal knowledge graphs from the parsed triplets.